In [ ]:
!pip install ipywidgets pypdf matplotlib

In [10]:
import os
import re
import shutil
import subprocess
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink

from pypdf import PdfReader

import matplotlib.pyplot as plt

In [12]:
def get_pdf_page_count(pdf_path):
    reader = PdfReader(str(pdf_path))
    return len(reader.pages)


def compress_pdf_ghostscript_with_progress(
    input_path,
    output_path,
    pdf_setting="/ebook",
    progress_bar=None,
    status_html=None,
    log_output=None
):
    """
    Ghostscript로 PDF를 압축하면서 페이지 단위 진행 상황을 UI에 반영합니다.
    """
    gs_executable = shutil.which("gs")
    if gs_executable is None:
        raise RuntimeError(
            "Ghostscript(gs)를 찾을 수 없습니다. 터미널에서 `brew install ghostscript` 후 다시 시도하세요."
        )

    total_pages = get_pdf_page_count(input_path)

    if progress_bar is not None:
        progress_bar.min = 0
        progress_bar.max = max(total_pages, 1)
        progress_bar.value = 0

    if status_html is not None:
        status_html.value = f"압축 준비 중... (총 {total_pages}페이지)"

    # -dQUIET 제거: 진행 로그를 읽기 위해 필요
    cmd = [
        gs_executable,
        "-sDEVICE=pdfwrite",
        "-dCompatibilityLevel=1.4",
        f"-dPDFSETTINGS={pdf_setting}",
        "-dNOPAUSE",
        "-dBATCH",
        f"-sOutputFile={str(output_path)}",
        str(input_path),
    ]

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    page_pattern = re.compile(r"Page\s+(\d+)")
    processed_pages = 0
    progress_points = []

    for line in process.stdout:
        line = line.strip()

        match = page_pattern.search(line)
        if match:
            processed_pages = int(match.group(1))
            processed_pages = min(processed_pages, total_pages)
            progress_points.append(processed_pages)

            if progress_bar is not None:
                progress_bar.value = processed_pages

            if status_html is not None:
                percent = (processed_pages / total_pages * 100) if total_pages else 0
                status_html.value = (
                    f"압축 중... {processed_pages}/{total_pages} 페이지 "
                    f"({percent:.1f}%)"
                )

        if log_output is not None and line:
            with log_output:
                print(line)

    return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)

    if progress_bar is not None:
        progress_bar.value = total_pages

    if status_html is not None:
        status_html.value = f"압축 완료 ({total_pages}/{total_pages} 페이지)"

    return {
        "total_pages": total_pages,
        "progress_points": progress_points
    }

In [13]:
upload = widgets.FileUpload(
    accept=".pdf",
    multiple=False,
    description="PDF 업로드"
)

setting_dropdown = widgets.Dropdown(
    options=[
        ("용량 최대 절감 (/screen)", "/screen"),
        ("균형형 (/ebook)", "/ebook"),
        ("화질 우선 (/printer)", "/printer"),
        ("고화질 (/prepress)", "/prepress"),
    ],
    value="/ebook",
    description="압축옵션"
)

run_button = widgets.Button(
    description="압축 실행",
    button_style="success"
)

progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="진행률",
    bar_style=""
)

status_html = widgets.HTML("대기 중")

output = widgets.Output()
chart_output = widgets.Output()
log_output = widgets.Output(layout={"border": "1px solid #ddd", "max_height": "200px", "overflow": "auto"})

In [14]:
def get_uploaded_file(upload_widget):
    value = upload_widget.value

    if not value:
        return None, None

    if isinstance(value, dict):
        uploaded = list(value.values())[0]
        return uploaded["name"], uploaded["content"]

    if isinstance(value, tuple):
        uploaded = value[0]
        return uploaded["name"], uploaded["content"]

    raise TypeError(f"지원하지 않는 upload.value 타입: {type(value)}")


def draw_progress_chart(progress_points, total_pages):
    with chart_output:
        clear_output(wait=True)

        if total_pages <= 0:
            print("그래프를 표시할 수 없습니다.")
            return

        if not progress_points:
            progress_points = [0]

        x = list(range(1, len(progress_points) + 1))
        y = progress_points

        plt.figure(figsize=(8, 4))
        plt.plot(x, y, marker="o")
        plt.xlabel("업데이트 횟수")
        plt.ylabel("처리된 페이지 수")
        plt.title("압축 진행 현황")
        plt.ylim(0, total_pages)
        plt.grid(True)
        plt.show()


def draw_size_chart(original_size_mb, compressed_size_mb):
    with chart_output:
        clear_output(wait=True)

        labels = ["원본", "압축본"]
        values = [original_size_mb, compressed_size_mb]

        plt.figure(figsize=(6, 4))
        plt.bar(labels, values)
        plt.ylabel("파일 크기 (MB)")
        plt.title("압축 전/후 크기 비교")
        plt.grid(axis="y")
        plt.show()


def on_run_clicked(b):
    with output:
        clear_output()
    with chart_output:
        clear_output()
    with log_output:
        clear_output()

    progress_bar.value = 0
    progress_bar.bar_style = ""
    status_html.value = "작업 시작"

    try:
        file_name, content = get_uploaded_file(upload)

        if file_name is None:
            with output:
                print("먼저 PDF 파일을 드래그 앤 드롭하거나 업로드하세요.")
            status_html.value = "파일 없음"
            progress_bar.bar_style = "warning"
            return

        input_path = Path(file_name).name
        stem = Path(file_name).stem
        suffix = Path(file_name).suffix or ".pdf"
        output_path = f"{stem}_compressed{suffix}"

        with open(input_path, "wb") as f:
            f.write(content)

        with output:
            print(f"원본 저장: {input_path}")
            print(f"압축 옵션: {setting_dropdown.value}")

        status_html.value = "페이지 수 확인 중..."

        result = compress_pdf_ghostscript_with_progress(
            input_path=input_path,
            output_path=output_path,
            pdf_setting=setting_dropdown.value,
            progress_bar=progress_bar,
            status_html=status_html,
            log_output=log_output
        )

        total_pages = result["total_pages"]
        progress_points = result["progress_points"]

        # 진행 그래프 표시
        draw_progress_chart(progress_points, total_pages)

        original_size = os.path.getsize(input_path) / 1024 / 1024
        compressed_size = os.path.getsize(output_path) / 1024 / 1024
        reduction = (1 - compressed_size / original_size) * 100 if original_size > 0 else 0

        with output:
            print(f"압축 완료: {output_path}")
            print(f"원본 크기: {original_size:.2f} MB")
            print(f"압축 크기: {compressed_size:.2f} MB")
            print(f"감소율: {reduction:.1f}%")
            display(FileLink(output_path, result_html_prefix="다운로드: "))

        # 마지막엔 크기 비교 그래프로 교체하고 싶으면 아래 줄만 유지
        draw_size_chart(original_size, compressed_size)

        progress_bar.bar_style = "success"
        status_html.value += " ✅"

    except subprocess.CalledProcessError as e:
        progress_bar.bar_style = "danger"
        status_html.value = "Ghostscript 실행 실패"
        with output:
            print("Ghostscript 실행 중 오류가 발생했습니다.")
            print(e)

    except Exception as e:
        progress_bar.bar_style = "danger"
        status_html.value = "오류 발생"
        with output:
            print("오류가 발생했습니다.")
            print(e)


run_button.on_click(on_run_clicked)

In [15]:
display(
    widgets.VBox([
        widgets.HTML("<h3>PDF 압축기 (드래그 앤 드롭 + Ghostscript)</h3>"),
        widgets.HTML("<p>PDF를 끌어다 놓고 압축을 실행하면 진행률과 결과 그래프를 볼 수 있습니다.</p>"),
        upload,
        setting_dropdown,
        run_button,
        progress_bar,
        status_html,
        widgets.HTML("<h4>압축 결과</h4>"),
        output,
        widgets.HTML("<h4>그래프</h4>"),
        chart_output,
        widgets.HTML("<h4>진행 로그</h4>"),
        log_output,
    ])
)